# 💠installations and imports

In [7]:
import pdb
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
#from .utils import *
import pdb
import matplotlib.pyplot as plt
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from timm.models.layers import DropPath, trunc_normal_
from timm.models.registry import register_model
#from timm.models.layers.helpers import to_2tuple
from timm.layers.helpers import to_2tuple
#from pyheatmap.heatmap import HeatMap
import random
#from pyheatmap.heatmap import HeatMap
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

In [5]:
#!pip install timm

# 💠  previous codes :  AxialAttention

In [8]:
#from .utils import *
import torch.nn as nn


class qkv_transform(nn.Conv1d):
    """Conv1d for qkv_transform"""

In [9]:
def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv3d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


In [10]:
class GroupNorm(nn.GroupNorm):
    """
    Group Normalization with 1 group.
    Input: tensor in shape [B, C, H, W]
    """
    def __init__(self, num_channels, **kwargs):
        super().__init__(1, num_channels, **kwargs)


In [12]:
class AxialAttention_dynamic(nn.Module):
    def __init__(self, in_planes, out_planes, groups=8, kernel_size=56,
                 stride=1, bias=False, width=False, length = False):
        assert (in_planes % groups == 0) and (out_planes % groups == 0)
        super(AxialAttention_dynamic, self).__init__()
        self.in_planes = in_planes
        self.out_planes = out_planes
        self.groups = groups
        self.group_planes = out_planes // groups
        self.kernel_size = kernel_size
        self.stride = stride
        self.bias = bias
        self.width = width
        self.length = length
        # Multi-head self attention
        self.qkv_transform = qkv_transform(in_planes, out_planes * 2, kernel_size=1, stride=1,
                                           padding=0, bias=False)
        self.bn_qkv = nn.BatchNorm1d(out_planes * 2)
        self.bn_similarity = nn.BatchNorm2d(groups * 3)
        self.bn_output = nn.BatchNorm1d(out_planes * 2)

        # Priority on encoding

        ## Initial values

        self.f_qr = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_kr = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_sve = nn.Parameter(torch.tensor(0.1),  requires_grad=False)
        self.f_sv = nn.Parameter(torch.tensor(1.0),  requires_grad=False)


        # Position embedding
        self.relative = nn.Parameter(torch.randn(self.group_planes * 2, kernel_size * 2 - 1), requires_grad=True)
        query_index = torch.arange(kernel_size).unsqueeze(0)
        key_index = torch.arange(kernel_size).unsqueeze(1)
        relative_index = key_index - query_index + kernel_size - 1
        self.register_buffer('flatten_index', relative_index.view(-1))
        if stride > 1:
            self.pooling = nn.AvgPool3d(3, stride=(2,2,2),padding=1)

        self.reset_parameters()
        # self.print_para()

    def forward(self, x): # N, C, L, H, W
        if self.length:
            x = x.permute(0, 3, 4, 1, 2) # N, H, W, C, L
        else:
          if self.width:
            x = x.permute(0, 2, 3, 1, 4)  # N, L, H, C, W
          else:
            x = x.permute(0, 2, 4, 1, 3)  # N, L, W, C, H
        N, L, W, C, H = x.shape
        
        x = x.contiguous().view(N * W * L, C, H)

        # Transformations
        qkv = self.bn_qkv(self.qkv_transform(x))
        q, k, v = torch.split(qkv.reshape(N * L * W, self.groups, self.group_planes * 2, H), [self.group_planes // 2, self.group_planes // 2, self.group_planes], dim=2)

        # Calculate position embedding
        all_embeddings = torch.index_select(self.relative, 1, self.flatten_index).view(self.group_planes * 2, self.kernel_size, self.kernel_size)
        q_embedding, k_embedding, v_embedding = torch.split(all_embeddings, [self.group_planes // 2, self.group_planes // 2, self.group_planes], dim=0)
        qr = torch.einsum('bgci,cij->bgij', q, q_embedding)
        kr = torch.einsum('bgci,cij->bgij', k, k_embedding).transpose(2, 3)
        qk = torch.einsum('bgci, bgcj->bgij', q, k)


        # multiply by factors
        qr = torch.mul(qr, self.f_qr)
        kr = torch.mul(kr, self.f_kr)

        stacked_similarity = torch.cat([qk, qr, kr], dim=1)
        stacked_similarity = self.bn_similarity(stacked_similarity).view(N * W * L, 3, self.groups, H, H).sum(dim=1)
        #stacked_similarity = self.bn_qr(qr) + self.bn_kr(kr) + self.bn_qk(qk)
        # (N, groups, H, H, W)
        similarity = F.softmax(stacked_similarity, dim=3)
        sv = torch.einsum('bgij,bgcj->bgci', similarity, v)
        sve = torch.einsum('bgij,cij->bgci', similarity, v_embedding)

        # multiply by factors
        sv = torch.mul(sv, self.f_sv)
        sve = torch.mul(sve, self.f_sve)

        stacked_output = torch.cat([sv, sve], dim=-1).view(N * L * W, self.out_planes * 2, H)
        output = self.bn_output(stacked_output).view(N, L, W, self.out_planes, 2, H).sum(dim=-2)

        if self.length:
            output = output.permute(0, 3, 4, 1, 2)
        else:
          if self.width:
            output = output.permute(0, 3, 1, 2, 4)  # N, W, C, H
          else:
            output = output.permute(0, 3, 1, 4, 2)
        
        if self.stride > 1:
            output = self.pooling(output)

        return output

        def reset_parameters(self):
            self.qkv_transform.weight.data.normal_(0, math.sqrt(1. / self.in_planes))
            #nn.init.uniform_(self.relative, -0.1, 0.1)
            nn.init.normal_(self.relative, 0., math.sqrt(1. / self.group_planes))

In [13]:
class AxialBlock_dynamic(nn.Module):
    expansion = 2

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None, kernel_size=56, length=16):
        super(AxialBlock_dynamic, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.))
        # Both self.conv2 and self.downsample layers downsample the input when stride != 1
        self.conv_down = conv1x1(inplanes, width)
        self.bn1 = norm_layer(4,width)
        self.hight_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size=kernel_size)
        self.width_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size=kernel_size, width=True)
        self.length_block = AxialAttention_dynamic(width, width, groups=groups, kernel_size = length, stride=stride, length=True)
        self.conv_up = conv1x1(width, planes * self.expansion)
        self.bn2 = norm_layer(4,planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv_down(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.hight_block(out)
        out = self.width_block(out)
        out = self.length_block(out)
        
        out = self.relu(out)

        out = self.conv_up(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        out = self.relu(out)

        return out


In [ ]:
class ResAxialAttentionUNet3D(nn.Module):
    # model = ResAxialAttentionUNet(AxialBlock_dynamic, [1, 2, 4, 1], s= 0.125, **kwargs)
    def __init__(self, block, layers, num_classes=4, zero_init_residual=True,
                 groups=8, width_per_group=64, replace_stride_with_dilation=None,
                 norm_layer=None, s=0.125, img_size=128, img_depth = 128,imgchan=3):
        super(ResAxialAttentionUNet3D, self).__init__()
        if norm_layer is None:
            norm_layer = nn.GroupNorm
        self._norm_layer = norm_layer

        self.inplanes = int(128 * s)
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError("replace_stride_with_dilation should be None "
                             "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv3d(imgchan, self.inplanes, kernel_size=7, stride=(2,2,2), padding=3,
                               bias=False)
        self.conv2 = nn.Conv3d(self.inplanes, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv3 = nn.Conv3d(128, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = norm_layer(4,self.inplanes)
        self.bn2 = norm_layer(4,128)
        self.bn3 = norm_layer(4,self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.AvgPool3d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, int(128 * s), layers[0], stride=1, kernel_size=(img_size // 4),length = img_depth//4)
        self.layer2 = self._make_layer(block, int(256 * s), layers[1], stride=2, kernel_size=(img_size // 4),
                                       dilate=replace_stride_with_dilation[0],length = img_depth//4)
        self.layer3 = self._make_layer(block, int(512 * s), layers[2], stride=2, kernel_size=(img_size // 8),
                                       dilate=replace_stride_with_dilation[1],length = img_depth//8)
        #self.layer4 = self._make_layer(block, int(1024 * s), layers[3], stride=1, kernel_size=(img_size // 16),
        #                               dilate=replace_stride_with_dilation[2],length = img_depth//16)
        #self.channel = conv1x1(int(64 * s),int(128 * s) , 1)
        # Decoder
        self.decoder1 = nn.Conv3d(int(1024 * 2 * s), int(1024 * 2 * s), kernel_size=3, stride=(2,2,2), padding=1)
        self.decoder2 = nn.Conv3d(int(1024  * s), int(1024 * s), kernel_size=3, stride=2, padding=1)
        self.decoder3 = nn.Conv3d(int(1024 * s), int(512 * s), kernel_size=3, stride=1, padding=1)
        self.decoder4 = nn.Conv3d(int(512 * s), int(256 * s), kernel_size=3, stride=1, padding=1)
        self.decoder5 = nn.Conv3d(int(256 * s), int(128 * s), kernel_size=3, stride=1, padding=1)
        self.decoder6 = nn.Conv3d(int(128 * s), int(64 * s), kernel_size=3, stride=1, padding=1)
        self.adjust = nn.Conv3d(int(64 * s), num_classes, kernel_size=1, stride=1, padding=0)
        

        self.soft = nn.Softmax(dim=1)

    def _make_layer(self, block, planes, blocks, kernel_size=56, stride=1, dilate=False, length=16):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(4,planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, groups=self.groups,
                            base_width=self.base_width, dilation=previous_dilation,
                            norm_layer=norm_layer, kernel_size=kernel_size, length = length))
        self.inplanes = planes * block.expansion
        if stride != 1:
            kernel_size = kernel_size // 2
            length = length // 2
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer, kernel_size=kernel_size, length = length))

        return nn.Sequential(*layers)

    def _forward_impl(self, x):

        # AxialAttention Encoder
        # pdb.set_trace()

        
      x_slice = self.conv1(x)
      x_slice = self.bn1(x_slice)
        
      x_slice = self.relu(x_slice)
      x_slice = self.conv2(x_slice)
      x_slice = self.bn2(x_slice)
      x_slice = self.relu(x_slice)
      x_slice = self.conv3(x_slice)
      x_slice = self.bn3(x_slice)
      x_slice = self.relu(x_slice)
      x_bfpool = x_slice
      x_slice = self.pool(x_slice)

      x1 = self.layer1(x_slice)

      x2 = self.layer2(x1)

      x3 = self.layer3(x2)
      #x4 = self.layer4(x3)

      #x_slice = F.relu(F.interpolate(self.decoder1(x4), scale_factor=(2, 2, 2), mode='nearest'))

      #x_slice = torch.add(x_slice, x4)
      x_slice = F.relu(F.interpolate(self.decoder2(x3), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x3)
      x_slice = F.relu(F.interpolate(self.decoder3(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x2)
      x_slice = F.relu(F.interpolate(self.decoder4(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = torch.add(x_slice, x1)
      x_slice = F.relu(F.interpolate(self.decoder5(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      #print(x_slice.shape)
      #print(x_bfpool.shape)
      #x_bfpool = self.channel(x_bfpool)
      #x_slice = torch.add(x_slice, x_bfpool)
      x_slice = F.relu(F.interpolate(self.decoder6(x_slice), scale_factor=(2, 2, 2), mode='trilinear'))
      x_slice = self.adjust(F.relu(x_slice))
      # pdb.set_trace()
        
      
      return x_slice

    def forward(self, x):
        return self._forward_impl(x)


# 💠 model

In [ ]:
'''
 فاز ۱: Encoder Pipeline ✅ شامل:

         ConvStem برای استخراج low-level features
         SelfAxialAttention برای مدالیته‌های مستقل
         CrossAxialAttention برای تعامل بین مدالیته‌ها
         MCCABlock در سطوح مختلف
         DualBranchEncoder برای دو مسیر T1/T1Gd و T2/FLAIR
         BottleneckBlock برای ادغام نهایی
 '''

## ✅ Encoder block

In [ ]:
''' ساختار کلی ورودی
x = {
    "t1":    (B, 1, D, H, W),
    "t1gd":  (B, 1, D, H, W),
    "t2":    (B, 1, D, H, W),
    "flair": (B, 1, D, H, W),
}
'''

In [ ]:
# برای استخراج ویژگی‌های اولیه
# ورودی (B, C=1, D=155, H=224, W=224)

import torch
import torch.nn as nn

class ConvStem(nn.Module):
    """
    ماژول Convolutional Stem برای استخراج ویژگی‌های اولیه با دو مرحله downsampling.
    """

    def __init__(self, in_channels: int = 1, base_channels: int = 32, return_all: bool = False):
        super().__init__()
        self.return_all = return_all

        self.conv1 = self._make_block(in_channels, base_channels, stride=1)
        self.conv2 = self._make_block(base_channels, base_channels, stride=2)
        self.conv3 = self._make_block(base_channels, base_channels, stride=2)

    def _make_block(self, in_ch, out_ch, stride):
        return nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x0 = self.conv1(x)   # resolution: 1x
        x1 = self.conv2(x0)   # resolution: 1/2
        x2 = self.conv3(x1)   # resolution: 1/4
        return (x0, x1, x2) if self.return_all else x2

'''
# داریم T1  استفاده ازش :فرض کنیم تصویر 
dummy = torch.randn(1, 1, 155, 224, 224)  # (B, C=1, D, H, W)

stem = ConvStem(in_channels=1, base_channels=32)
x0, x1, x2 = stem(dummy)

print(f"x0 (1x):   {x0.shape}")  # [1, 32, 155, 224, 224]
print(f"x1 (1/2):  {x1.shape}")  # [1, 32, 77, 112, 112]
print(f"x2 (1/4):  {x2.shape}")  # [1, 32, 38, 56, 56]


x0:	برای skip connection در decoder
x1:	(اختیاری) برای encoder عمیق‌تر یا multi-scale
x2:	ورودی به self-modal axial attention block
'''

In [14]:
'''وقتی فقط یک مدالیته (مثلاً T1) رو داریم، می‌خوایم یاد بگیریم
فقط روی یک محور در هر بار attention می‌زنیم:
اول D (عمق) → بعد H (ارتفاع) → بعد W (عرض)
'''

import torch
import torch.nn as nn
import torch.nn.functional as F

class AxialAttention(nn.Module):
    def __init__(self, dim, num_heads=4, axis='H'):
        """
        dim: تعداد کانال‌های ورودی/خروجی
        num_heads: تعداد headها در attention
        axis: یکی از ['D', 'H', 'W'] برای محور مورد توجه
        """
        super().__init__()
        assert axis in ['D', 'H', 'W'], "Axis must be 'D', 'H', or 'W'"
        self.axis = axis
        self.num_heads = num_heads
        self.dim = dim
        
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        '''برای scaling امتیاز attention‌ها
        (یعنی ضرب داخلی Q و K تقسیم بر ریشه بعد)
        جلوگیری از بزرگ شدن مقدار dot product بین Q و K
        اگر مقدار dot زیاد بشه →‌ softmax خیلی شارپ می‌شه → گرادیان ناپایدار'''

        self.qkv = nn.Linear(dim, dim * 3)
        '''این لایه ورودی رو به Q، K، V تبدیل می‌کنه
        خروجی‌اش: [batch, length, 3 * dim]
        بعداً با chunk(3) جداش می‌کنیم
        dim = 32 , nn.Linear(32, 96)  qkv = self.qkv(x)   shape: (B, L, 96)
        q, k, v = qkv.chunk(3, dim=-1)   هرکدوم (B, L, 32)'''
        
        
        self.proj = nn.Linear(dim, dim)
        # بعد از attention، خروجی رو دوباره به فضای ویژگی اصلی می‌بره
        self.norm = nn.LayerNorm(dim)
        # برای normalizing بعد از residual connection

    def forward(self, x):
        B, C, D, H, W = x.shape

        if self.axis == 'H':
            x = x.permute(0, 2, 4, 3, 1).contiguous().view(B * D * W, H, C)
            '''هر sequence طول H داره
                تعداد sequenceها: B * D * W 
                (یعنی برای هر اسلایس، هر ستون عمودی)
                نتیجه: آماده‌ایم تا sequenceهایی به شکل (sequence_len, embed_dim) داشته باشیم که attention بخواد'''
            
        elif self.axis == 'W':
            x = x.permute(0, 2, 3, 4, 1).contiguous().view(B * D * H, W, C)
        elif self.axis == 'D':
            x = x.permute(0, 3, 4, 2, 1).contiguous().view(B * H * W, D, C)

        qkv = self.qkv(x).chunk(3, dim=-1)
        '''x رو به Q، K، V تقسیم می‌کنیم،
        هرکدوم شکلش:[batch*, length, dim]'''

        q, k, v = [t.reshape(t.shape[0], t.shape[1], self.num_heads, -1).transpose(1, 2) for t in qkv]
        #q, k, v = map(lambda t: t.reshape(batch, length, num_heads, head_dim).transpose(1, 2), qkv)


        attn = (q @ k.transpose(-2, -1)) * self.scale
        #برای هر head، ضرب داخلی Q و K برای مقایسه similarity
        
        attn = attn.softmax(dim=-1)
        #بعد با softmax نرمال‌سازی
        
        #بازگشت به فضای اصلی
        out = (attn @ v).transpose(1, 2).reshape(x.shape[0], x.shape[1], C)
        #reshape(batch*, length, dim)
        '''q: (B, h, L, d) = (2, 4, 10, 8)
        k: (2, 4, 10, 8)
        v: (2, 4, 10, 8)
        attn = softmax(q @ k.T)  → shape: (2, 4, 10, 10)
        (attn @ v)  → (2, 4, 10, 8)
        .transpose(1, 2) → (2, 10, 4, 8)
        .reshape(2, 10, 32)  # یعنی 4 * 8 = dim
        '''
        out = self.proj(out)
        '''این لایه برای ترکیب کردن headهای attention هست.
        هر head فقط بخشی از featureها رو یاد می‌گیره. این لایه کمک می‌کنه:
        همه رو باهم fuse کنیم'''
        out = self.norm(out + x)
        # با residual به input اضافه می‌کنیم

        
        # برگرداندن به شکل اولیه (B, C, D, H, W)
        if self.axis == 'H':
            out = out.view(B, D, W, H, C).permute(0, 4, 1, 3, 2)
        elif self.axis == 'W':
            out = out.view(B, D, H, W, C).permute(0, 4, 1, 2, 3)
        elif self.axis == 'D':
            out = out.view(B, H, W, D, C).permute(0, 4, 3, 1, 2)

        return out

'''
# فرض: خروجی stem برای یک مدالیته
x2 = torch.randn(1, 32, 38, 56, 56)

self_modal = SelfModalAxialBlock(dim=32, num_heads=4)
out = self_modal(x2)

print(out.shape)
 باید باشه: (1, 32, 38, 56, 56)
'''

In [ ]:
class SelfModalAxialBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        """
        dim: تعداد کانال‌های ورودی/خروجی
        num_heads: تعداد headهای attention
        """
        super().__init__()
        self.attn_d = AxialAttention(dim, num_heads=num_heads, axis='D')
        self.attn_h = AxialAttention(dim, num_heads=num_heads, axis='H')
        self.attn_w = AxialAttention(dim, num_heads=num_heads, axis='W')

    def forward(self, x):
        x = self.attn_d(x)
        x = self.attn_h(x)
        x = self.attn_w(x)
        return x


In [ ]:
# هر پیکسل از مدالیته A (مثلاً T1) با کدوم پیکسل‌ها از مدالیته B (مثلاً T1Gd) مرتبطه
# x_q: Tensor با شکل (B, C, D, H, W)  ← مدالیته A (مثلاً T1)
# x_kv: Tensor با شکل (B, C, D, H, W) ← مدالیته B (مثلاً T1Gd)

import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossAxialAttention(nn.Module):
    def __init__(self, dim, num_heads=4, axis='H'):
        """
        Cross-Axial Attention:
        Q ← از x_q
        K, V ← از x_kv
        axis: یکی از 'D', 'H', 'W'
        """
        super().__init__()
        assert axis in ['D', 'H', 'W'], "Axis must be 'D', 'H', or 'W'"
        self.axis = axis
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.dim = dim

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_q, x_kv):
        # Input: x_q, x_kv shape = [B, C, D, H, W]
        B, C, D, H, W = x_q.shape

        def reshape(x, axis):
            if axis == 'D':
                return x.permute(0, 3, 4, 2, 1).contiguous().view(B * H * W, D, C)
            elif axis == 'H':
                return x.permute(0, 2, 4, 3, 1).contiguous().view(B * D * W, H, C)
            elif axis == 'W':
                return x.permute(0, 2, 3, 4, 1).contiguous().view(B * D * H, W, C)

        def reverse(out, axis, shape):
            B, C, D, H, W = shape
            if axis == 'D':
                return out.view(B, H, W, D, C).permute(0, 4, 3, 1, 2)
            elif axis == 'H':
                return out.view(B, D, W, H, C).permute(0, 4, 1, 3, 2)
            elif axis == 'W':
                return out.view(B, D, H, W, C).permute(0, 4, 1, 2, 3)

        xq_r = reshape(x_q, self.axis)
        xkv_r = reshape(x_kv, self.axis)

        # Q, K, V
        Q = self.q_proj(xq_r)
        K = self.k_proj(xkv_r)
        V = self.v_proj(xkv_r)

        Bq, Lq, _ = Q.shape
        Q = Q.view(Bq, Lq, self.num_heads, self.head_dim).transpose(1, 2)  # (Bq, h, Lq, d)
        K = K.view(Bq, Lq, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(Bq, Lq, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        out = (attn @ V).transpose(1, 2).reshape(Bq, Lq, self.dim)
        out = self.out_proj(out)
        out = self.norm(out + xq_r)  # residual

        return reverse(out, self.axis, (B, C, D, H, W))

''' 
:مثال عددی از عملکرد
فرض ورودی:
            x_q.shape = x_kv.shape = (1, 4, 8, 16, 16)  # (B, C, D, H, W)
            dim = 4
            num_heads = 2
            head_dim = 2
مرحله 1: reshape برای محور H
            xq_r = x_q.permute(0, 2, 4, 3, 1).contiguous().view(1 * 8 * 16, 16, 4)
                 → shape: (128, 16, 4)
مرحله 2: Q, K, V projection
            Q = Linear(4→4)(xq_r) = (128, 16, 4)        128 sequence, طول 16, embed_dim=4
            K = Linear(4→4)(xkv_r) = (128, 16, 4)
            V = Linear(4→4)(xkv_r) = (128, 16, 4)

            عملکرد nn.Linear(4, 4) یعنی:
                            برای هر پیکسل به‌تنهایی (shape: (4,))، داریم:
                            output_pixel = W @ input_pixel + b
                            که:
                            W: ماتریس وزن سایز (4×4)
                            b: بایاس سایز (4,)
                            input_pixel: یه بردار 4تایی (مثلاً [0.3, 0.1, -0.6, 0.7])
                            output_pixel: بردار جدید 4تایی با وزن‌دهی جدید
                            یعنی:
                                    روی هر یک از اون 128×16 پیکسل، یک Linear مستقل اعمال می‌شه.
                                    وزن‌ها مشترک هستن، ولی ورودی‌ها متفاوته.
                                    ✅ بنابراین: هر توکن (پیکسل) وارد یک فضای جدید وزن‌دهی‌شده می‌شه
                                        
مرحله 3: تقسیم به headها
            Q.view(128, 16, 2, 2).transpose(1, 2) → (128, 2, 16, 2)
مرحله 4: attention score
            out = attn @ V → shape: (128, 2, 16, 2)
مرحله 5: attention روی V
            out = attn @ V → shape: (128, 2, 16, 2)
مرحله 6: ادغام headها (concatenate)
            out.transpose(1, 2).reshape(128, 16, 4)  # ← concat headها
مرحله 7: projection نهایی
            self.out_proj = Linear(4 → 4)
            این لایه می‌گه: «اوکی، تو از 2 یا 4 تا head اطلاعات گرفتی، حالا همه‌ی اون headها رو با وزن‌های جدید باهم ترکیب کن».
            یعنی برای هر پیکسل (طول 16)، یک بردار 4تایی داریم.
            out_proj برای هر بردار 4تایی:        
            یه ماتریس وزن جدید یاد می‌گیره    
            شبیه به conv1x1 در CNN عمل می‌کنه (برای ترکیب channelها)
مرحله 8: reshape back
            out.view(1, 8, 16, 16, 4).permute(0, 4, 1, 3, 2)
            → shape: (1, 4, 8, 16, 16)



In [ ]:
class CrossModalAxialBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn_d = CrossAxialAttention(dim, num_heads=num_heads, axis='D')
        self.attn_h = CrossAxialAttention(dim, num_heads=num_heads, axis='H')
        self.attn_w = CrossAxialAttention(dim, num_heads=num_heads, axis='W')
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_q, x_kv):
        # سه محور: D, H, W
        out_d = self.attn_d(x_q, x_kv)
        out_h = self.attn_h(out_d, x_kv)
        out_w = self.attn_w(out_h, x_kv)

        return out_w


In [ ]:
'''
x_a: مدالیته اول (مثلاً T1)
x_b: مدالیته دوم (مثلاً T1Gd)

ما بعد از self-modal و cross-modal، برای هر مدالیته دو نسخه داریم:
برای T1:
        xa: ویژگی‌هایی که از خود T1 (Self-Axial) استخراج کردیم
        xa_cross: ویژگی‌هایی که T1 از T1Gd یاد گرفته (Cross-Attention)
همین‌طور برای T1Gd:
        xb
        xb_cross
        
حالا سؤال:  چطور این دو رو با هم ترکیب کنیم؟
همیشه xa_cross بهتره؟ همیشه xa ؟ یا بسته به مکان تصمیم بگیریم؟
1. بدون gating: ساده‌ترین حالت
        xa_out = xa + xa_cross
        یعنی می‌گیم: «هر دو مهمن → جمعشون کن»
        ❌ اما تو همه جا اهمیت برابری نمی‌دن

2. با gating: adaptive fusion
        gate_a = sigmoid(conv3d(xa))  → shape: (B, C, D, H, W) ∈ (0, 1)
        برای هر پیکسل یه وزن می‌سازه که:
                اگر gate نزدیک به 1: یعنی به cross-attention بیشتر اعتماد کن
                اگر gate نزدیک به 0: یعنی به self-attention بیشتر اعتماد کن
🔹 چی کار می‌کنه؟
        Conv3d(1×1×1) فقط یک لایه weighting یاد می‌گیره → شبیه به attention mask   
        Sigmoid خروجی رو بین 0 تا 1 نگه می‌داره → مناسب برای وزن‌دهی
'''

class MCCABlock(nn.Module):
    def __init__(self, dim, num_heads=4, use_gating=True):
        super().__init__()
        
        self.self_a = SelfModalAxialBlock(dim, num_heads)
        self.self_b = SelfModalAxialBlock(dim, num_heads)
        #هرکدوم شامل 3 لایه Axial Attention در محور D, H, W برای خودشون هستن.

        self.cross_a = CrossModalAxialBlock(dim, num_heads)
        self.cross_b = CrossModalAxialBlock(dim, num_heads)
        #چه ارتباط ساختاری بین این دو هست استفاده از کلاس CrossAxialAttention

        self.use_gating = use_gating
        if use_gating:
            self.gate_a = nn.Sequential(
                nn.Conv3d(dim, dim, kernel_size=1),
                nn.Sigmoid()
            )
            self.gate_b = nn.Sequential(
                nn.Conv3d(dim, dim, kernel_size=1),
                nn.Sigmoid()
            )
            #gate_a یاد می‌گیره که چقدر به xa_cross اعتماد کنه و چقدر به xa

    def forward(self, x_a, x_b):
        # x_a, x_b: مدالیته‌های هم‌گروه (مثلاً T1, T1Gd)

        # Self-modal axial attention
        xa = self.self_a(x_a)
        xb = self.self_b(x_b)

        # Cross-modal attention
        xa_cross = self.cross_a(xa, xb)
        xb_cross = self.cross_b(xb, xa)

        # Fusion (gated or simple add/mul)
        if self.use_gating:
            gate_a = self.gate_a(xa)
            gate_b = self.gate_b(xb)
            xa_out = gate_a * xa_cross + (1 - gate_a) * xa
            xb_out = gate_b * xb_cross + (1 - gate_b) * xb
        else:
            xa_out = xa + xa_cross
            xb_out = xb + xb_cross

        return xa_out, xb_out

#  این بلاک رو معمولاً در ۳ سطح مختلف resolution استفاده می‌کنی

In [ ]:
'''
| مرحله                     | شرح                                                   |
| ------------------------- | ------------------------------------------- ------- |
| `ConvStem`                | استخراج ویژگی‌های اولیه و downsampling تا resolution 1/4    |
| `SelfAxial` (درون MCCA)   | یادگیری وابستگی محوری داخل هر مدالیته                         |
| `CrossAxial` (درون MCCA)  | یادگیری وابستگی ساختاری بین دو مدالیته مرتبط                     |
| `Gating Fusion` (در MCCA) | ترکیب تطبیقی اطلاعات local و contextual                  |
'''
'''
✳️ هدف کلی این ماژول:
یادگیری ویژگی‌های سطح بالا از هر دو شاخه‌ی مدالیته‌ای:
شاخه 1: T1 + T1Gd
شاخه 2: T2 + FLAIR
با استفاده از:
    ConvStem برای استخراج low-level features
    سه مرحله MCCA برای یادگیری self + cross attention
    Downsampling تدریجی برای abstraction
    
'''


class DualBranchEncoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=32, num_heads=4, use_gating=True):
        super().__init__()
        # ConvStem برای هر مدالیته به صورت مستقل
        self.stem_t1     = ConvStem(in_channels, base_channels)
        self.stem_t1gd   = ConvStem(in_channels, base_channels)
        self.stem_t2     = ConvStem(in_channels, base_channels)
        self.stem_flair  = ConvStem(in_channels, base_channels)
        '''
        هر کدوم یک stem ساده شامل:
        Conv3d → BN → ReLU
        دو بار downsampling
        → خروجی نهایی: resolution = 1/4 از تصویر اصلی
        '''

        # سه سطح MCCA برای هر شاخه
        self.mcca1_branch1 = MCCABlock(base_channels, num_heads, use_gating)
        self.mcca2_branch1 = MCCABlock(base_channels, num_heads, use_gating)
        self.mcca3_branch1 = MCCABlock(base_channels, num_heads, use_gating)

        self.mcca1_branch2 = MCCABlock(base_channels, num_heads, use_gating)
        self.mcca2_branch2 = MCCABlock(base_channels, num_heads, use_gating)
        self.mcca3_branch2 = MCCABlock(base_channels, num_heads, use_gating)

        # Downsampling بین سطوح
        self.downsample = nn.AvgPool3d(kernel_size=2, stride=2)

    def forward(self, t1, t1gd, t2, flair):
        # خروجی سطح 2 هر stem که تصویر 1/4 شده
        _, _, t1_l2     = self.stem_t1(t1)
        _, _, t1gd_l2   = self.stem_t1gd(t1gd)
        _, _, t2_l2     = self.stem_t2(t2)
        _, _, flair_l2  = self.stem_flair(flair)

        # MCCA سطح 1
        t1_l2, t1gd_l2     = self.mcca1_branch1(t1_l2, t1gd_l2)
        t2_l2, flair_l2    = self.mcca1_branch2(t2_l2, flair_l2)

        # Downsample → سطح 1/8
        t1_l3     = self.downsample(t1_l2)
        t1gd_l3   = self.downsample(t1gd_l2)
        t2_l3     = self.downsample(t2_l2)
        flair_l3  = self.downsample(flair_l2)

        # MCCA سطح 2
        t1_l3, t1gd_l3     = self.mcca2_branch1(t1_l3, t1gd_l3)
        t2_l3, flair_l3    = self.mcca2_branch2(t2_l3, flair_l3)

        # Downsample → سطح 1/16
        t1_l4     = self.downsample(t1_l3)
        t1gd_l4   = self.downsample(t1gd_l3)
        t2_l4     = self.downsample(t2_l3)
        flair_l4  = self.downsample(flair_l3)

        # MCCA سطح 3
        t1_l4, t1gd_l4     = self.mcca3_branch1(t1_l4, t1gd_l4)
        t2_l4, flair_l4    = self.mcca3_branch2(t2_l4, flair_l4)

        # خروجی نهایی ۳ سطح هر شاخه برای bottleneck و skip-connection
        return {
            't1':    [t1_l2, t1_l3, t1_l4],
            't1gd':  [t1gd_l2, t1gd_l3, t1gd_l4],
            't2':    [t2_l2, t2_l3, t2_l4],
            'flair': [flair_l2, flair_l3, flair_l4],
        }

'''
✅ این خروجی‌ها مستقیماً می‌رن برای:
    bottleneck encoder (در resolution پایین)
    skip connection در TCFC decoder

    برای هر مدالیته، سه سطح feature map داریم
    برای استفاده در bottleneck و decoder (TCFC)
'''


## ✅ Buttleneck Block

In [ ]:
'''
ورودی bottleneck از خروجی DualBranchEncoder داریم:

dict = {
  't1': [lvl2, lvl3, lvl4],
  't1gd': [...],
  't2': [...],
  'flair': [...]
}
ما فقط به lvl4 هر شاخه نیاز داریم
'''

'''
nn.LayerNorm([C]) در PyTorch انتظار داره داده به شکل [batch, ..., C] باشه
اما ورودی ما 5D هست: [B, C, D, H, W]
→ پس باید channel رو بیاریم آخر

فرض کن:
        B=1, C=4, D=2, H=2, W=2
        x.shape = (1, 4, 2, 2, 2)
        →  مجموعاً 8 voxel داریم، هرکدوم با 4 کانال
مرحله 1:
        x.view(B, C, -1) → (1, 4, 8)
مرحله 2:
        .transpose(1, 2) → (1, 8, 4)
        یعنی: هر voxel، بردار feature 4تایی داره → حالا می‌تونیم روی dim=-1 LayerNorm بزنیم
مرحله 3:
        self.norm(...) → LayerNorm روی هر feature vector (size=4)
مرحله 4:
        ✅ برگردوندن به شکل اصلی بعد از normalization
        .transpose(1, 2) → (1, 4, 8)
        .view(B, C, D, H, W) → (1, 4, 2, 2, 2)   
'''



class BottleneckBlock(nn.Module):
    def __init__(self, in_channels=32, out_channels=32, use_axial=False):
        super().__init__()
        self.use_axial = use_axial

        self.norm = nn.LayerNorm([out_channels * 4, 1, 1, 1])  # normalize over channels
        '''
        چون 4 مدالیته داریم (T1, T1Gd, T2, FLAIR)، موقع concat → channel = 4 * out_channels
        LayerNorm بر اساس تعداد کانال normalize می‌کنه
         در 3D، برخلاف BatchNorm که بر B, D, H, W کار می‌کنه، LayerNorm فقط روی channel عمل می‌کنه → مناسب‌تر برای attention-style معماری
        '''

        if use_axial:
            self.refine = SelfModalAxialBlock(out_channels * 4)
        else:
            self.refine = nn.Sequential(
                nn.Conv3d(out_channels * 4, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_channels),
                nn.ReLU(inplace=True)
            )
        '''
        ویژگی	               SelfAxialAttention	               Conv3D
        دید	                      global (در هر محور)	           local
        حافظه                    	              سنگین‌تر            	         سبک‌تر
        receptive field	      kernel sizeبی‌نهایت          در یک محور محدود به    
        کاربرد           	برای ویژگی‌های بلندبرد یا diffuse (مثل edema)        برای refinement محلی یا سریع
        '''

    def forward(self, x1, x2, x3, x4):
        x = torch.cat([x1, x2, x3, x4], dim=1)  # (B, 4C, D, H, W)
        B, C, D, H, W = x.shape

        x = self.norm(x.view(B, C, -1).transpose(1, 2)).transpose(1, 2).view(B, C, D, H, W)
        x = self.refine(x)
        return x  # (B, C, D, H, W)

'''
خروجی Bottleneck:
shape: (B, 32, D/16, H/16, W/16)
'''

bottleneck = BottleneckBlock(in_channels=32, out_channels=32)
x = bottleneck(t1_l4, t1gd_l4, t2_l4, flair_l4)


## ✅Decoder Block

In [ ]:
'''
فاز ۲: Decoder و TCFC (بازسازی و هم‌ترازی)                         

        | ترتیب اجرا | نام ماژول                 | توضیح عملکرد کلی                                                                                |
| ---------- | ------------------------- | ----------------------------------------------------------------------------------------------- |
| 1️⃣        | `UpsampleBlock`           | افزایش اندازه ویژگی‌ها از bottleneck (مثلاً از 1/16 به 1/8) با interpolation یا ConvTranspose3d |
| 2️⃣        | `TCFCBlock`               | تطبیق ویژگی‌های attention-based (از encoder) با CNN-based (از decoder) در هر سطح resolution     |
| 3️⃣        | `DecoderBlock`            | پردازش و refinement ویژگی‌ها بعد از ترکیب TCFC، معمولاً شامل Conv3d + BN + ReLU                 |
| 4️⃣        | `FinalSegHead`            | لایه نهایی segmentation: یک لایه Conv3d(1x1x1) برای پیش‌بینی کلاس هر voxel                      |
| 5️⃣        | (اختیاری) `FinalUpsample` | اگر resolution نهایی segmentation پایین باشه (مثلاً 1/2)، این ماژول full-res نهایی تولید می‌کنه |

'''

In [ ]:
'''
افزایش resolution ویژگی‌ها در مسیر decoder
(B, C, D/16, H/16, W/16) ⟶ (B, C, D/8, H/8, W/8)

|          معایب                 |       مزایا                            |                           روش              |
| -------------------------------------- | -----------------------------------  | -------------------------------- |
| `Interpolation` (trilinear) + `Conv3d` | سبک‌تر، بدون checkerboard artifacts     | کیفیت پایین‌تر نسبت به transpose         |
| `ConvTranspose3d`                      | می‌تونه یاد بگیره چطور upsample کنه | سنگین‌تر، مستعد artifact                       |

'''
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, scale_factor=2, mode='trilinear'):
        super().__init__()
        self.scale_factor = scale_factor
        self.mode = mode
        self.proj = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
        '''
        چون interpolation خودش یاد نمی‌گیره چیزی
        projection (با Conv1x1) می‌تونه ویژگی‌ها رو اصلاح کنه، کانال‌ها رو کاهش/افزایش بده، و non-linearity وارد کنه
        '''

    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.scale_factor, mode=self.mode, align_corners=False)
        x = self.proj(x)
        return x


upsample = UpsampleBlock(32, 32)
x = torch.randn(1, 32, 8, 14, 14)
y = upsample(x)
# → y.shape = (1, 32, 16, 28, 28)


In [ ]:
'''
ترکیب و تطبیق دو نوع feature که ذاتاً متفاوت‌اند:
|                    مسیر                        |                 ویژگی‌ها           |
| ---------------------------------------- | -------------------------------      |
| 🔵 Decoder (CNN-based) → `F_main`        | local, پایین‌برد global context          |
| 🟣 Encoder (Attention-based) → `F_trans` | long-range, حافظه‌دار، غیرمحلی            |

F_main  = از decoder (CNN)
F_trans = از skip (encoder attention)

چقدر از F_trans حفظ کنیم؟ چقدر با F_main جایگزین کنیم؟
این تصمیم، voxel-wise و محور به محور (X/Y/Z) یاد گرفته می‌شه.

|          مرحله                                 |                         توضیح                                 |
| ------------------------------------- | ----------------------------------------------------------------     |
| 1️⃣ Projection در X/Y/Z برای هر ورودی |              با Conv3d(kernel=(1,x,y)) ویژگی‌ها رو در هر محور استخراج می‌کنیم      |
| 2️⃣ Concatenate و Fuse                | ترکیب projectionها از دو مسیر                                             |
| 3️⃣ Attention Gate (Sigmoid)          | یاد می‌گیره کدوم ویژگی‌ها رو retain یا suppress کنه                             |
| 4️⃣ ترکیب تطبیقی نهایی                |                             خروجی: `F_fused = Gate * F_trans + F_main`    |

'''
'''
در U-Netهای کلاسیک، skip connectionها فقط یه concat ساده بین encoder و decoder هستن.
اما توی مدل شما:
        مسیر encoder حاوی ویژگی‌های attention-based هست (مثلاً Axial یا Transformer) 
        مسیر decoder از CNN تشکیل شده و ویژگی‌هاش local و مبتنی بر همسایگیه
        ❗ این دوتا کاملاً متفاوتن، و اگر مستقیم جمع/concat بشن، منجر به mismatch و noise می‌شن.
 هدف TCFC:
تطبیق هوشمندانه‌ی ویژگی‌های attention-based از encoder با ویژگی‌های CNN-based از decoder تا بتونن به درستی با هم ترکیب بشن.

TCFC چه کار می‌کنه؟
✅ 1. Projection محور-محور
        از هر ورودی (F_main, F_trans) سه projection جداگانه گرفته می‌شه:
        |     عملیات   |     محور                                     |
        | ----------  | ------------------------------------------ |
        | Z (عمق)     | `Conv3d(kernel=(3,1,1))` ← نگاه عمودی         |
        | Y (ارتفاع)    | `Conv3d(kernel=(1,3,1))` ← نگاه بالا-پایین        |
        | X (عرض)    | `Conv3d(kernel=(1,1,3))` ← نگاه چپ-راست       |
        
        این کار باعث می‌شه که ویژگی‌ها به‌صورت جهت‌دار (anisotropic) در هر محور spatial تحلیل بشن.
        بسیار مفید برای داده‌های 3D پزشکی که تومورها ممکنه فقط در یک جهت گسترش داشته باشن.
                شکل تنسور ورودی:
                فرض کن feature map ورودی (مثلاً F_main) این شکله:
                x.shape = (B=1, C=1, D=5, H=3, W=3)
                Slice 0:   Slice 1:   Slice 2:   Slice 3:   Slice 4:
                [[0,0,0]   [[1,1,1]   [[2,2,2]   [[3,3,3]   [[4,4,4]
                 [0,0,0]    [1,1,1]    [2,2,2]    [3,3,3]    [4,4,4]
                 [0,0,0]]   [1,1,1]]   [2,2,2]]   [3,3,3]]   [4,4,4]]
                هر slice یک سطح از تصویر MRI هست (محور Z).
                
                ✅ کانولوشن جهت‌دار Z → Conv3d(kernel=(3,1,1))
                یعنی kernel فقط در محور Z (depth) حرکت می‌کنه.
                و در هر قدم، یک voxel از slice بالا، وسط، پایین رو با هم ترکیب می‌کنه.
                    Kernel window:
                     ↓
                Slice i-1
                Slice  i
                Slice i+1
                     ↑
                ✅ پس برای مرکزیت voxel (i, h, w) ما فقط 3 مقدار از همین مختصات در 3 slice مختلف داریم → فقط به رابطه بین اسلایس‌ها حساسیم.
                بقیه بعدها (H, W) بدون تغییر عبور می‌کنن.
                
                ✅ کانولوشن جهت‌دار Y → Conv3d(kernel=(1,3,1))
                این یکی فقط روی محور ارتفاع (Y) کار می‌کنه:
                
                ✅ کانولوشن جهت‌دار X → Conv3d(kernel=(1,1,3))
                این یکی فقط چپ، مرکز، راست رو توی همون slice و همون ردیف (Y) با هم ترکیب می‌کنه.

                چرا جدا؟
                چون:
                Feature در Z ممکنه پراکندگی داشته باشه (مثلاً edema)
                در X ممکنه تقارن داشته باشه (مثلاً نواحی سفید-خاکستری)
                در Y مثلاً midline shift مهمه (تورم مغز)

                برای مثال:
                اگر تومور در محور Z پخش شده (بین sliceها)، projection Z اون رو شناسایی می‌کنه.
                اگر shift مغزی داریم، projection Y اون رو سریع تشخیص می‌ده.
                اگر عدم تقارن داریم (مثلاً در هِمی‌سفر راست مغز)، projection X بهش حساسه.

✅ 2. Fusion
        fusion = torch.cat([proj_main_x, ..., proj_trans_z], dim=1)
        6 مسیر projection → با هم concat می‌شن → یک Tensor با 6×C channel
✅ 3. Attention Gate
        gate = Conv3d(6C → C) → BN → ReLU → Conv3d(C → C) → Sigmoid
        یعنی TCFC نمی‌گه “فقط محور X مهمه”
        بلکه می‌گه: “از اطلاعاتی که projection X/Y/Z ساختن، یک gate بساز که بگه هر voxel باید چقدر از encoder بگیره و چقدر از decoder”
        یعنی هر voxel می‌تونه weight مخصوص خودش رو بگیره
این gate یاد می‌گیره که:
        «در این voxel (در این محور)، چقدر به attention اعتماد کنم؟ و چقدر به decoder؟»
✅ 4. Adaptive ترکیب:
        F_fused = gate * F_trans + (1 - gate) * F_main

'''

class TCFCBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.proj_main_x = nn.Conv3d(channels, channels, kernel_size=(1, 1, 3), padding=(0, 0, 1))
        self.proj_main_y = nn.Conv3d(channels, channels, kernel_size=(1, 3, 1), padding=(0, 1, 0))
        self.proj_main_z = nn.Conv3d(channels, channels, kernel_size=(3, 1, 1), padding=(1, 0, 0))

        self.proj_trans_x = nn.Conv3d(channels, channels, kernel_size=(1, 1, 3), padding=(0, 0, 1))
        self.proj_trans_y = nn.Conv3d(channels, channels, kernel_size=(1, 3, 1), padding=(0, 1, 0))
        self.proj_trans_z = nn.Conv3d(channels, channels, kernel_size=(3, 1, 1), padding=(1, 0, 0))

        self.fuse = nn.Sequential(
            nn.Conv3d(channels * 6, channels, kernel_size=1),
            nn.BatchNorm3d(channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels, channels, kernel_size=1),
            nn.Sigmoid()  # Gate
        )

    def forward(self, F_main, F_trans):
        # Projections
        p_main_x = self.proj_main_x(F_main)
        p_main_y = self.proj_main_y(F_main)
        p_main_z = self.proj_main_z(F_main)

        p_trans_x = self.proj_trans_x(F_trans)
        p_trans_y = self.proj_trans_y(F_trans)
        p_trans_z = self.proj_trans_z(F_trans)

        # Fusion
        fusion = torch.cat([p_main_x, p_main_y, p_main_z, p_trans_x, p_trans_y, p_trans_z], dim=1)
        gate = self.fuse(fusion)

        # Adaptive feature calibration
        F_fused = gate * F_trans + F_main
        return F_fused

'''
خروجی:
F_fused: shape = (B, C, D, H, W)
→ هدایت‌شده توسط gate، حاوی ترکیب آگاهانه‌ای از ویژگی‌های attention و CNN
'''

'''معمولاً وقتی از encoder میای (skip connection) و decoder رو upsample کردی، ممکنه شکلشون فرق کنه:
مثلاً:
| tensor                 | شکل                                                    |
| ---------------------- | ------------------------------------------------------ |
| `F_main` (از decoder)  | (B, C, 32, 64, 64) ← از upsample اومده                 |
| `F_trans` (از encoder) | (B, C, 31, 63, 63) ← ممکنه برش خورده یا downsample شده |
✅ راه‌حل TCFC:
در مرحله قبل از TCFC (یعنی داخل decoder)، باید:

یا قبل از ورودی به TCFC، encoder features رو resize کنیم:
یا F_main رو crop کنیم تا با F_trans match شه
    
📌 این بخشی از pipeline decoder هست، نه داخل TCFC خودش.
یعنی: ما تضمین می‌کنیم که ورودی TCFC دو feature map با شکل یکسان باشن.
'''


In [ ]:
'''
بعد از اینکه با TCFC ویژگی‌های attention و CNN از encoder و decoder هماهنگ شدن، DecoderBlock:
        🟡ویژگی‌ها رو refine می‌کنه

        ✏️ وقتی شما یه feature map رو:
                از encoder گرفته باشی (و ممکنه coarse و noisy باشه)
                یا از پایین decoder upsample کرده باشی (و جزئیات spatial کم شده باشه)
                این feature map نیاز به «پردازش بیشتر» داره تا:
                با resolution جدید هماهنگ بشه
                noise‌ها حذف بشن
                context غنی‌تر بشه
                مرزها شارپ‌تر بشن
                ✅ این دقیقاً کاریه که DecoderBlock انجام می‌ده.
        🟡 به مدل کمک می‌کنه semantic gap بین feature mapهای coarse و fine رو پر کنه
                 اول بفهمیم semantic gap چیه؟
                                 | سطح شبکه                   | نوع ویژگی‌ها                                                                  |
                | -------------------------- | ----------------------------------------------------------------------------- |
                | Encoder levels پایین (1/4) |      لبه، بافت، local جزئیات دقیق ← **low-level features**                         |
                | Bottleneck (1/16)          | مفهوم کلی تومور، کلاس‌بندی، ساختار بزرگ ← **high-level features**             |
                | Decoder                    | باید بین این دوتا رو وصل کنه (هم مرز بفهمه، هم مفهوم) ← **middle-high-level** |

                ⚠️ مشکل:
                وقتی skip connection بزنی (مثلاً از encoder سطح 1/4 به decoder 1/4)، ممکنه:
                        
                        ویژگی‌های encoder فقط local و خام باشن
                        ویژگی‌های decoder خیلی abstract باشن (contextual)
                        → ترکیب مستقیمشون ممکنه noisy یا بی‌معنا باشه.

                 DecoderBlock چطور حلش می‌کنه؟
                            با داشتن 2 یا 3 لایه Conv3D + BN + ReLU پشت سر هم، DecoderBlock به مدل اجازه می‌ده:
                            خودش تصمیم بگیره کدوم ویژگی‌ها رو نگه داره، کدوم رو حذف کنه
                            اطلاعات fine-grained از encoder رو با semantic info از bottleneck تطبیق بده

        🟡 امکان non-linearity، regularization، و contextualization بیشتر فراهم می‌کنه


         

|    دلیل           |                  راه حل پیشنهادی                              |
| ----------------------------  | ------------------------------------------ |
| 2× Conv3D (3×3×3)             | رایج‌ترین ساختار برای refinement                  |
| BN + ReLU بعد از هر conv        | پایداری و non-linearity                        |
| optional residual connection  | کمک به گرادیان و تطبیق با encoder features        |

🧠 حالت‌های قابل تنظیم:
|     اثر                         |                       پارامتر                                |
| ----------------------------   | --------------------------------------------------------- |
| `in_channels ≠ out_channels`   | برای وقتی که resolution عوض شده یا کانال‌ها کاهش داده شدن            |
| `use_residual=True`            | وقتی بخوای stability یا تطبیق سریع‌تر ویژگی‌ها داشته باشی               |



'''

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels=None, use_residual=False):
        super().__init__()
        if out_channels is None:
            out_channels = in_channels
        #  اگر تعداد کانال خروجی مشخص نشده باشه، یعنی بخوایم بدون تغییر ابعاد کار کنیم

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)

        self.use_residual = use_residual
        if use_residual and in_channels != out_channels:
            self.res_conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)
        #  اگر بخوای stability بیشتر و training راحت‌تر داشته باشی، می‌تونی از residual connection استفاده کنی:
        # ولی اگه in_channels ≠ out_channels، اول باید یه conv تطبیق‌دهنده بزنی.

    def forward(self, x):
        identity = x

        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))

        if self.use_residual:
            identity = self.res_conv(identity) if hasattr(self, 'res_conv') else identity
            x += identity

        return x



# فرض: بعد از TCFC
refiner = DecoderBlock(in_channels=32, out_channels=32)
refined_feat = refiner(fused_feat)
# → خروجی‌ای داریم که هم ساختار مرز رو می‌فهمه، هم semantic class اون ناحیه رو

In [ ]:
'''
🎯 هدف DecoderStage
ترکیب ۳ عمل اصلی:

🔼 Upsample کردن خروجی مرحله قبل (مثلاً از 1/16 به 1/8)
🔁 ترکیب آگاهانه با skip connection (TCFCBlock)
🧪 Refinement نهایی ویژگی‌ها (DecoderBlock)

✅ ورودی‌ها:
|  توضیح   |                    نام                                          |
| ------ | --------------------------------------------------------------- |
| `x`    | feature map ورودی از bottleneck یا مرحله قبلی decoder                 |
| `skip` | feature map skip connection از encoder، در همون resolution جدید     |

دلیل                 	مزیت
--------------------------------------------------------------
کاملاً  مستقل          	می‌تونی چند بار توی decoder استفاده‌اش کنی
کاملاً تطبیق‌پذیر	       در encoder هایی با تعداد سطوح متفاوت قابل تنظیمه
قابل تست	          هر DecoderStage رو می‌تونی جدا تست و visualize کن

'''

class DecoderStage(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = UpsampleBlock(in_channels, out_channels)
        self.tcfc = TCFCBlock(out_channels)
        self.refine = DecoderBlock(out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)         # 🔼 افزایش resolution
        x = self.tcfc(x, skip)       # 🔁 ترکیب با skip connection
        x = self.refine(x)           # 🧪 پالایش ویژگی‌ها
        return x



x = DecoderStage(64, 32)(x, skip_lvl3)
x = DecoderStage(32, 16)(x, skip_lvl2)
x = DecoderStage(16, 8)(x, skip_lvl1)


In [ ]:
'''
✅ : هدف Decoder
گرفتن feature فشرده‌شده از Bottleneck + skip connection از Encoder
و بازسازی map نهایی segmentation، به‌صورت voxel-wise prediction

ساختار کلی decoder که:
|           وظیفه                  |                بخش                               |
| ---------------------------- | --------------------------------------------------- |
| 🔁 چند `DecoderStage`        | بازسازی تدریجی resolution (مثلاً از 1/16 ⟶ 1/2)           |
| 🎯 `FinalSegHead`           | تولید خروجی نهایی segmentation با تعداد کلاس دلخواه            |
| 🆙 (اختیاری) `FinalUpsample`   | اگه بخوای resolution نهایی full-size بشه                 |

✅ ورودی‌ها به Decoder
|شکل        |            توضیح            |        نام                               |
| ------- | -------------------------- | --------------------------------------- |
| `x`     | `(B, C, D/16, H/16, W/16)` | خروجی Bottleneck                        |
| `skips` | لیست طول 3                  | `[level3, level2, level1]` ← از encoder |
| هر skip | `(B, C, D/r, H/r, W/r)`    | مطابق resolution مرحله‌ی decoder مربوطه     |


'''

class Decoder(nn.Module):
    def __init__(self, channels=[64, 32, 16], out_classes=4, final_upsample=False):
        super().__init__()
        self.stages = nn.ModuleList([
            DecoderStage(channels[0], channels[1]),  # 1/16 ⟶ 1/8
            DecoderStage(channels[1], channels[2]),  # 1/8 ⟶ 1/4
            DecoderStage(channels[2], channels[2])   # 1/4 ⟶ 1/2
        ])
        '''
        🧠 این لیست ۳ مرحله داره، هر کدوم شامل:
            UpsampleBlock ← افزایش resolution
            TCFCBlock ← ترکیب با encoder (skip)
            DecoderBlock ← پالایش ویژگی‌ها
            '''
        
        self.seg_head = nn.Conv3d(channels[2], out_classes, kernel_size=1)
        '''
        🎯 آخرین لایه کانولوشن:
                یک کانولوشن ساده 1×1×1 برای تبدیل آخرین feature map به logits برای segmentation
                خروجی: (B, num_classes, D/2, H/2, W/2)
                '''
        self.final_upsample = final_upsample
        # اگر بخوای resolution نهایی رو برگردونی به full size، با این فلگ کنترلش می‌کنی
        

    def forward(self, x, skips):
        for stage, skip in zip(self.stages, skips):
            x = stage(x, skip)   # ← یک مرحله‌ی decoder
            '''
            🧠 توی هر مرحله:
                resolution رو افزایش می‌دی (upsample)
                skip connection همون سطح رو باهاش ترکیب می‌کنی (TCFC)
                نتیجه رو refine می‌کنی (DecoderBlock)

                Stage 1: x (1/16) + skip (1/8) → out: (1/8)
                Stage 2: x (1/8) + skip (1/4) → out: (1/4)
                Stage 3: x (1/4) + skip (1/2) → out: (1/2)

            '''

        out = self.seg_head(x)   # 🎯 پیش‌بینی segmentation

        if self.final_upsample:
            out = F.interpolate(out, scale_factor=2, mode="trilinear", align_corners=False)
            
        # 📌 اگر بخوای output map رو به full resolution برسونی (برابر با input MRI)، این upsampling اجرا می‌شه:
        # (B, 4, 80, 160, 160) → (B, 4, 160, 320, 320)


        return out



# فرض: خروجی bottleneck
x = (B, 64, D/16, H/16, W/16)

# و خروجی skipهای encoder (3 سطح):
skips = [skip3, skip2, skip1]  # (1/8), (1/4), (1/2)

decoder = Decoder(channels=[64, 32, 16], out_classes=4)
seg_logits = decoder(x, skips)


# 💠 SegmentationModel

In [ ]:
'''
✅ شامل این ماژول‌ها:
| بخش           | ماژول                                           |
| ------------- | ---------------------------------------------- |
| 🔹 Encoder    | `DualBranchEncoder` (با CKD + Axial Attention)  |
| 🔸 Bottleneck | `BottleneckBlock` (ترکیب 4 مدالیته)                |
| 🔺 Decoder    | `Decoder` (3 DecoderStage + FinalSegHead)      |

✅ ورودی مدل:
T1, T1Gd, T2, FLAIR → (B, 1, D, H, W)
(همه مدالیته‌ها به صورت 1-channel جداگانه)

✅ خروجی:
segmentation logits → (B, num_classes, D, H, W) (یا 1/2 رزولوشن)
'''

class HybridSegmentationModel(nn.Module):
    def __init__(self, in_channels=1, base_channels=32, out_classes=4):
        super().__init__()
        '''
        🔸 ورودی‌ها:
                in_channels: تعداد کانال برای هر مدالیته MRI (مثلاً 1 برای grayscale)
                base_channels: تعداد کانال پایه (برای ConvStem و تمام ماژول‌ها)
                out_classes: تعداد کلاس segmentation (مثلاً 4: background, edema, tumor, core)
         '''

        self.encoder = DualBranchEncoder(
            in_channels=in_channels,
            base_channels=base_channels,
            num_heads=4,
            use_gating=True
        )
        '''
        🧠 ماژول DualBranchEncoder قبلاً ساختی:
                دارای دو شاخه: T1/T1Gd و T2/FLAIR
                هر شاخه شامل:
                ConvStem
                3 سطح MCCA (Self+Cross Axial)
                خروجی برای هر مدالیته: [lvl1, lvl2, lvl3]
            '''

        self.bottleneck = BottleneckBlock(
            in_channels=base_channels,
            out_channels=base_channels,
            use_axial=False  # یا True بسته به نیاز
        )
        '''
        📌 ترکیب خروجی 4 مدالیته در سطح آخر encoder (resolution 1/16):
            | مدالیته   | سطح خروجی     |
            | --------- | ------------- |
            | T1, T1Gd  | `lvl3` = 1/16 |
            | T2, FLAIR | `lvl3` = 1/16 |

            🔸 وظیفه Bottleneck:
                    concat → normalization → refinement
                    با axial یا conv ساده بسته به use_axial

        '''

        self.decoder = Decoder(
            channels=[base_channels * 2, base_channels, base_channels // 2],
            out_classes=out_classes,
            final_upsample=True  # اگر خروجی full-size بخوای
        )
        '''
        🧠 ماژول Decoder هم قبلاً ساختی:
        
        شامل 3 مرحله:
                DecoderStage(1/16 → 1/8)
                DecoderStage(1/8 → 1/4)
                DecoderStage(1/4 → 1/2)
        
        و سپس FinalSegHead برای تولید خروجی segmentation
        اگر final_upsample=True باشه، خروجی می‌تونه full resolution بشه
        '''

    def forward(self, t1, t1gd, t2, flair):
        # 🔹 Encoder
        enc_outs = self.encoder(t1, t1gd, t2, flair)
        '''
        📦 خروجی enc_outs یک دیکشنریه:
                {
                  't1':    [lvl1, lvl2, lvl3],  # resolution: 1/4, 1/8, 1/16
                  't1gd':  [...],
                  't2':    [...],
                  'flair': [...]
                }
        '''

        # 🔸 Bottleneck
        x = self.bottleneck(
            enc_outs['t1'][2],
            enc_outs['t1gd'][2],
            enc_outs['t2'][2],
            enc_outs['flair'][2]
        )
        '''
        🔍 ورودی‌های bottleneck ← آخرین سطح encoder (1/16 resolution)
        📌 این بخش feature mapهای 4 مدالیته رو در محور channel ترکیب می‌کنه و refinement انجام می‌ده
        '''

        # استخراج skip connectionها برای decoder (از سطح 1 تا 3)
        skips = [
            torch.cat([enc_outs['t1'][1], enc_outs['t1gd'][1]], dim=1),   # level 3: 1/8
            torch.cat([enc_outs['t2'][1], enc_outs['flair'][1]], dim=1),  # level 2: 1/4
            torch.cat([enc_outs['t1'][0], enc_outs['t2'][0]], dim=1)      # level 1: 1/2
        ]

        # 🔺 Decoder
        seg_logits = self.decoder(x, skips)

        return seg_logits


model = HybridSegmentationModel()
out = model(t1, t1gd, t2, flair)  # → shape: (B, 4, D, H, W)



In [15]:
def gated_3d(pretrained=False, **kwargs):
    model = ResAxialAttentionUNet3D(AxialBlock_dynamic, [1, 2, 4, 1], s= 0.25, **kwargs)
    return model

In [ ]:
'''
🔥 فاز ۳: Training Pipeline + Self-Distillation
🎯 هدف:
        آموزش مدل با بهره‌گیری از regularization و warm-up supervision
📦 موارد کلیدی:
        WarmUpDLB: Deep supervision روی feature maps encoder و decoder
        SelfDistillLoss: اتکا به prediction خود مدل برای regularizing (بدون teacher)
        Training script: loss functions, optimizer, data loader, evaluation
'''

In [ ]:
'''
🧪 فاز ۴: Evaluation & Ablation
🎯 هدف:
    تحلیل تاثیر هر بخش از معماری	
    
| آیتم                             | هدف تست                                 S |
| -------------------------------- | -------------------------------------- |
| Ablation: w/o axial attention    | اهمیت axial attention                   |
| Ablation: w/o gating in fusion   | بررسی adaptive weighting                |
| Comparison: CKD vs proposed      | مقایسه مستقیم طراحی شما با مدل CKD اصلی         |
| Metrics: Dice, HD95, Sensitivity | ارزیابی کیفیت segmentation                  |

'''

In [ ]:
'''
📊 فاز ۵: Visualization + Reporting

| مورد                   | ابزار یا خروجی                        |
| ---------------------- | ------------------------------------- |
| Feature Map Visualizer | بررسی خروجی MCCABlock و Bottleneck    |
| Segmentation Viewer    | نمایش prediction vs GT                |
| نمودار training loss   | برای تحلیل overfit یا underfit        |
| گزارش مقاله            | توضیح معماری + ablation + مقایسه عددی |

'''

In [ ]:
'''
| فاز   | توضیح                 | وضعیت            |
| ----- | --------------------- | ---------------- |
| فاز 1 | Encoder + Bottleneck  | ✅ انجام شد       |
| فاز 2 | Decoder + TCFC        | 🔜 در حال شروع   |
| فاز 3 | Training + Loss       | ⏳ بعد از decoder |
| فاز 4 | Ablation + Eval       | 📊 پس از آموزش   |
| فاز 5 | Visualization + Paper | 📝 آخر مسیر      |

'''